In [1]:
!nvidia-smi

Fri Feb 13 07:11:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   29C    P0             49W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!sudo python3 -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 52.8 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0


In [2]:
from getpass import getpass
import os

hf = getpass("Paste your HUGGINGFACE_HUB_TOKEN: ")
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf

Paste your HUGGINGFACE_HUB_TOKEN:  ········


In [4]:
!pip install -U pip
!pip install -U vllm openai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 107.5 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 241.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 205.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 194.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 69.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 147.2 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 178.0 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 176.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 219.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 177.2 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 185.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
import vllm
print("vLLM version:", vllm.__version__)

vLLM version: 0.15.1


In [10]:
!pip install "numpy<1.28" --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 188.8 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
import subprocess, shlex, os, time, requests

MODEL_ID = "google/medgemma-1.5-4b-it"
HOST = "127.0.0.1"
PORT = 8000

# Get token from environment
hf_token = os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not hf_token:
    raise ValueError("HUGGINGFACE_HUB_TOKEN is not set")

# Build command
VLLM_CMD = f"""
vllm serve {MODEL_ID}
--host {HOST}
--port {PORT}
--dtype=float32
--gpu-memory-utilization 0.90
--trust-remote-code
"""

# Create env explicitly
env = os.environ.copy()
env["HUGGINGFACE_HUB_TOKEN"] = hf_token
env["HF_TOKEN"] = "add-hf-token-here"  # some libs use this

# Start server with logs to file
log_file = open("vllm.log", "w")

proc = subprocess.Popen(
    shlex.split(VLLM_CMD),
    stdout=log_file,
    stderr=log_file,
    env=env
)

print("vLLM started with PID:", proc.pid)


base_url = f"http://{HOST}:{PORT}/v1"
for i in range(300):
    try:
        r = requests.get(f"{base_url}/models", timeout=2)
        if r.status_code == 200:
            print("vLLM server is ready")
            print(r.json())
            break
    except:
        time.sleep(1)
else:
    print("Server did not start. Check logs.")

vLLM started with PID: 28647


In [5]:
import requests, json

url = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "google/medgemma-1.5-4b-it",
    "messages": [
        {"role": "system", "content": "You are a helpful medical assistant."},
        {"role": "user", "content": "What does a chest X-ray show?"}
    ],
    "temperature": 0.2,
    "max_tokens": 256
}

response = requests.post(url, json=payload, timeout=120)
data = response.json()

print(data["choices"][0]["message"]["content"])

As a helpful medical assistant, I can provide you with some general information about what a chest X-ray shows. However, it's important to remember that I am an AI and cannot provide medical advice. A qualified healthcare professional should interpret the specific images and provide you with a diagnosis and treatment plan.

A chest X-ray is a common imaging test that uses X-rays to create pictures of the structures inside your chest. These structures include:

* **Lungs:** The main organs in the chest, responsible for taking in oxygen and releasing carbon dioxide.
* **Heart:** The muscular organ that pumps blood throughout the body.
* **Blood vessels:** The tubes that carry blood to and from the heart and lungs (like the aorta and pulmonary arteries/veins).
* **Ribs, bones, and muscles:** The supporting structures of the chest wall.
* **Trachea (windpipe):** The tube that carries air to the lungs.
* **Diaphragm:** The large muscle located at the bottom of the chest cavity that helps wi